# Case Study 01 — Credit Scoring: Retail Portfolio
## Notebook 01: Exploratory Data Analysis

**Author:** Erick Condoy · Economist (UNL) · Quant Researcher  
**Dataset:** UCI Default of Credit Card Clients (30,000 obs, Taiwan 2005)  
**Objective:** Understand the structure, quality, and predictive signals of the credit portfolio before modeling.

---

### Analytical Flow
1. Data ingestion & schema validation  
2. Missing values & data quality  
3. Target variable analysis (class imbalance)  
4. Univariate analysis — numeric & categorical features  
5. Bivariate analysis — features vs. default  
6. Correlation & multicollinearity screening  
7. Key findings summary


In [ ]:
# ── 0. Environment setup ──────────────────────────────────────────────────
import sys, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Paths (agnostic — no absolute OS paths)
DATA_DIR = Path('../data')
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

# Institutional plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.spines.left': True,
    'axes.spines.bottom': True,
    'axes.edgecolor': '#dcd9d5',
    'axes.labelcolor': '#28251d',
    'axes.titlepad': 10,
    'text.color': '#28251d',
    'xtick.color': '#7a7974',
    'ytick.color': '#7a7974',
    'font.family': 'DejaVu Sans',
    'font.size': 10,
    'axes.titlesize': 11,
    'figure.dpi': 120,
})

PALETTE = {'good': '#437a22', 'bad': '#a12c7b', 'neutral': '#01696f'}
print(f'numpy {np.__version__} | pandas {pd.__version__}')

## 1. Data Ingestion & Schema Validation

In [ ]:
# ── 1. Load data ──────────────────────────────────────────────────────────
# UCI dataset ships as Excel; try both naming conventions
candidates = list(DATA_DIR.glob('*.xls')) + list(DATA_DIR.glob('*.xlsx')) + list(DATA_DIR.glob('*.csv'))

if not candidates:
    raise FileNotFoundError(
        f'No data file found in {DATA_DIR}. '
        'Run: bash ../data/download.sh'
    )

fpath = candidates[0]
print(f'[INFO] Loading: {fpath.name}')

if fpath.suffix in ('.xls', '.xlsx'):
    df_raw = pd.read_excel(fpath, header=1)   # UCI Excel has 2-row header
else:
    df_raw = pd.read_csv(fpath)

# Defensive: drop the ID column if present
df_raw.columns = df_raw.columns.str.strip().str.upper().str.replace(' ', '_')
if 'ID' in df_raw.columns:
    df_raw = df_raw.drop(columns=['ID'])

# Rename target for clarity
TARGET = 'DEFAULT'
df_raw = df_raw.rename(columns={'DEFAULT.PAYMENT.NEXT.MONTH': TARGET,
                                 'DEFAULT_PAYMENT_NEXT_MONTH': TARGET})

print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# ── 1b. Schema introspection ──────────────────────────────────────────────
schema = pd.DataFrame({
    'dtype': df_raw.dtypes,
    'n_unique': df_raw.nunique(),
    'null_count': df_raw.isnull().sum(),
    'null_pct': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'sample_values': [df_raw[c].dropna().unique()[:3].tolist() for c in df_raw.columns]
})
schema

## 2. Missing Values & Data Quality

In [ ]:
# ── 2. Missing value heatmap ─────────────────────────────────────────────
missing_pct = df_raw.isnull().mean() * 100
missing_pct = missing_pct[missing_pct > 0]

if missing_pct.empty:
    print('[OK] No missing values detected. Dataset is complete.')
else:
    fig, ax = plt.subplots(figsize=(8, max(3, len(missing_pct) * 0.4)))
    missing_pct.sort_values().plot.barh(ax=ax, color=PALETTE['bad'], alpha=0.8)
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Value Rate by Feature')
    ax.axvline(5, color=PALETTE['neutral'], lw=1, ls='--', label='5% threshold')
    ax.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(REPORTS_DIR / 'fig_missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 2b. Outlier detection — IQR method on numeric cols ────────────────────
numeric_cols = df_raw.select_dtypes(include=np.number).columns.drop(TARGET, errors='ignore').tolist()

outlier_report = []
for col in numeric_cols:
    q1, q3 = df_raw[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_outliers = ((df_raw[col] < q1 - 3*iqr) | (df_raw[col] > q3 + 3*iqr)).sum()
    outlier_report.append({'feature': col, 'n_outliers': n_outliers,
                            'outlier_pct': round(n_outliers / len(df_raw) * 100, 2)})

pd.DataFrame(outlier_report).sort_values('n_outliers', ascending=False).head(10)

## 3. Target Variable Analysis

In [ ]:
# ── 3. Class imbalance ───────────────────────────────────────────────────
target_counts = df_raw[TARGET].value_counts()
default_rate = df_raw[TARGET].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
colors = [PALETTE['good'], PALETTE['bad']]
axes[0].bar(['Good (0)', 'Bad (1)'], target_counts.values, color=colors, alpha=0.85, width=0.5)
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(df_raw)*100:.1f}%)',
                ha='center', va='bottom', fontsize=9, color='#28251d')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
axes[0].set_ylim(0, target_counts.max() * 1.2)

# Donut
wedges, texts, autotexts = axes[1].pie(
    target_counts.values,
    labels=['Good', 'Bad'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    wedgeprops={'width': 0.55},
)
for t in autotexts: t.set_fontsize(10)
axes[1].set_title('Default Rate')

fig.suptitle(f'Target Variable: DEFAULT | Default Rate = {default_rate:.1f}%', fontsize=12, fontweight='semibold')
plt.tight_layout()
fig.savefig(REPORTS_DIR / 'fig_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Default rate: {default_rate:.2f}%')

## 4. Univariate Analysis

In [ ]:
# ── 4a. Demographic features ─────────────────────────────────────────────
cat_features = {
    'SEX': {1: 'Male', 2: 'Female'},
    'EDUCATION': {1: 'Grad school', 2: 'University', 3: 'High school', 4: 'Other', 5: 'Unknown', 6: 'Unknown'},
    'MARRIAGE': {0: 'Unknown', 1: 'Married', 2: 'Single', 3: 'Other'}
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (col, label_map) in zip(axes, cat_features.items()):
    counts = df_raw[col].map(label_map).value_counts()
    ax.bar(counts.index, counts.values, color=PALETTE['neutral'], alpha=0.8)
    ax.set_title(col)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=20)

fig.suptitle('Demographic Feature Distributions', fontsize=12, fontweight='semibold')
plt.tight_layout()
fig.savefig(REPORTS_DIR / 'fig_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 4b. Credit limit & age distributions ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Credit limit
axes[0].hist(df_raw['LIMIT_BAL'] / 1000, bins=50, color=PALETTE['neutral'], alpha=0.8, edgecolor='white', lw=0.5)
axes[0].set_xlabel('Credit Limit (NT$ thousands)')
axes[0].set_ylabel('Count')
axes[0].set_title('Credit Limit Distribution')
axes[0].axvline(df_raw['LIMIT_BAL'].median()/1000, color=PALETTE['bad'], lw=1.5, ls='--',
                label=f"Median: {df_raw['LIMIT_BAL'].median()/1000:.0f}K")
axes[0].legend(fontsize=8)

# Age
axes[1].hist(df_raw['AGE'], bins=40, color=PALETTE['neutral'], alpha=0.8, edgecolor='white', lw=0.5)
axes[1].set_xlabel('Age (years)')
axes[1].set_ylabel('Count')
axes[1].set_title('Age Distribution')
axes[1].axvline(df_raw['AGE'].median(), color=PALETTE['bad'], lw=1.5, ls='--',
                label=f"Median: {df_raw['AGE'].median():.0f}")
axes[1].legend(fontsize=8)

plt.tight_layout()
fig.savefig(REPORTS_DIR / 'fig_limit_age.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Bivariate Analysis — Features vs. Default

In [ ]:
# ── 5a. Default rate by categorical feature ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (col, label_map) in zip(axes, cat_features.items()):
    dr = df_raw.groupby(df_raw[col].map(label_map))[TARGET].mean() * 100
    bars = ax.bar(dr.index, dr.values, color=[PALETTE['bad'] if v > default_rate else PALETTE['good'] for v in dr.values],
                  alpha=0.8)
    ax.axhline(default_rate, color='#7a7974', lw=1.2, ls='--', label=f'Avg {default_rate:.1f}%')
    ax.set_title(f'Default Rate by {col}')
    ax.set_ylabel('Default Rate (%)')
    ax.tick_params(axis='x', rotation=20)
    ax.legend(fontsize=7)

fig.suptitle('Default Rate by Demographic Features', fontsize=12, fontweight='semibold')
plt.tight_layout()
fig.savefig(REPORTS_DIR / 'fig_default_by_demographics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5b. Credit limit vs. default (box plots) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col in zip(axes, ['LIMIT_BAL', 'AGE']):
    data_good = df_raw.loc[df_raw[TARGET] == 0, col]
    data_bad = df_raw.loc[df_raw[TARGET] == 1, col]
    bp = ax.boxplot([data_good, data_bad], patch_artist=True,
                    medianprops={'color': 'white', 'lw': 2},
                    whiskerprops={'color': '#dcd9d5'},
                    capprops={'color': '#dcd9d5'},
                    flierprops={'marker': 'o', 'alpha': 0.2, 'ms': 2})
    bp['boxes'][0].set_facecolor(PALETTE['good'])
    bp['boxes'][0].set_alpha(0.75)
    bp['boxes'][1].set_facecolor(PALETTE['bad'])
    bp['boxes'][1].set_alpha(0.75)
    ax.set_xticklabels(['Good (0)', 'Bad (1)'])
    ax.set_title(f'{col} by Default Status')
    ax.set_ylabel(col)

plt.tight_layout()
fig.savefig(REPORTS_DIR / 'fig_limit_age_vs_default.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5c. Payment status features — key predictors ──────────────────────────
pay_cols = [c for c in df_raw.columns if c.startswith('PAY_') and 'AMT' not in c]

default_by_pay = {col: df_raw.groupby(col)[TARGET].mean() * 100 for col in pay_cols[:6]}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for ax, col in zip(axes, list(default_by_pay.keys())):
    dr = default_by_pay[col]
    ax.bar(dr.index.astype(str), dr.values,
           color=[PALETTE['bad'] if v > default_rate else PALETTE['neutral'] for v in dr.values],
           alpha=0.8)
    ax.axhline(default_rate, color='#7a7974', lw=1, ls='--')
    ax.set_title(f'Default Rate by {col}')
    ax.set_xlabel('Payment Status')
    ax.set_ylabel('Default Rate (%)')
    ax.tick_params(axis='x', rotation=0, labelsize=8)

fig.suptitle('Default Rate by Payment History (PAY_0 to PAY_6)', fontsize=12, fontweight='semibold')
plt.tight_layout()
fig.savefig(REPORTS_DIR / 'fig_payment_status.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation & Multicollinearity Screening

In [ ]:
# ── 6. Correlation heatmap ───────────────────────────────────────────────
corr = df_raw[numeric_cols + [TARGET]].corr()

# Mask upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    corr, mask=mask, ax=ax,
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 7},
    linewidths=0.5, linecolor='#f7f6f2',
    cbar_kws={'shrink': 0.7}
)
ax.set_title('Pearson Correlation Matrix', fontsize=12, fontweight='semibold')
ax.tick_params(axis='both', labelsize=7)
plt.tight_layout()
fig.savefig(REPORTS_DIR / 'fig_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 6b. Top correlations with target ─────────────────────────────────────
target_corr = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(7, 6))
colors = [PALETTE['bad'] if v < 0 else PALETTE['good'] for v in target_corr.values]
ax.barh(target_corr.index, target_corr.values, color=colors, alpha=0.8)
ax.axvline(0, color='#dcd9d5', lw=1)
ax.set_xlabel('Pearson Correlation with DEFAULT')
ax.set_title('Feature Correlation with Target', fontweight='semibold')
plt.tight_layout()
fig.savefig(REPORTS_DIR / 'fig_target_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Findings Summary

In [ ]:
# ── 7. Summary statistics table ──────────────────────────────────────────
summary = df_raw.describe().T
summary['skew'] = df_raw[summary.index].skew().round(3)
summary['kurt'] = df_raw[summary.index].kurt().round(3)
summary

---

## EDA Findings

| Finding | Detail | Modeling Implication |
|---------|--------|---------------------|
| **Class imbalance** | ~22% default rate | Use class_weight='balanced' or SMOTE |
| **Payment history dominates** | PAY_0 most correlated with default | Key predictor — include WOE-transformed |
| **Credit limit is protective** | Higher limits → lower default rate | Non-linear — bin with WOE |
| **Multicollinearity in bill amounts** | BILL_AMT1–6 highly correlated | Consider PCA or drop redundant |
| **Education signal weak** | 'Unknown' category prevalent | Clean & merge unknown categories |
| **No missing values** | Complete dataset | No imputation required |

**Next step:** `02_feature_engineering.ipynb` — WOE/IV binning, fine/coarse classing, scorecard scaling.
